In [ ]:
import pandas as pd 
import numpy as np
import yfinance as yf
import statsmodels.api as sm

# --- CONFIGURAÇÃO ---
years = range(2014, 2025)
k = 5
modes = ["fast", "medium", "slow"]
centralities = ["central", "peripheral"]

# Função que calcula o beta
def beta_ols(rp, rm):
    # Garante que rm seja uma Series com nome
    rm = rm.squeeze().rename("rm")
    df = rp.to_frame("rp").join(rm, how="inner").dropna()
    
    if len(df) < 5: return np.nan # Proteção contra dados insuficientes
    
    y = df["rp"]
    X = sm.add_constant(df["rm"])
    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 5}
    )
    return model.params["rm"]

# Dicionário para armazenar os betas finais ou retornos
# Estrutura: results[year][k][centrality]
all_betas = {}

for year in years:
    print(f"Processando ano: {year}...")
    all_betas[year] = {}
    
    # 1. Baixando Benchmark (SP500) para o ano específico
    # Usamos start/end cobrindo o ano para bater com os retornos dos portfolios
    sp500 = yf.download(tickers="^GSPC", start=f"{year}-01-01", end=f"{year}-12-31", progress=False)
    sp500_dr = sp500["Close"].pct_change().dropna()
    
    # 2. Carregando retornos do ano
    try:
        returns = pd.read_parquet(f"../../data/02_clean/returns_new_{year}.parquet")
    except FileNotFoundError:
        print(f"Arquivo de retornos para {year} não encontrado. Pulando...")
        continue

    all_betas[year][k] = {}
    
    for centrality in centralities:
        # 3. Pegando metadados dos tickers (ajustado para k variável)
        try:
            metadata_path = f"../../data/07_portfolios_metadata/{centrality}_metadata_{year}_{k}.csv"
            tickers = pd.read_csv(metadata_path)["Ticker"]
            
            # Filtrar apenas tickers que existem no arquivo de retornos
            valid_tickers = [t for t in tickers if t in returns.columns]
            portfolio_returns = returns[valid_tickers].mean(axis=1) # Retorno médio do portfólio
            
            # 4. Calcular Beta
            beta_val = beta_ols(portfolio_returns, sp500_dr)
            all_betas[year][k][centrality] = beta_val
            
        except Exception as e:
            print(f"Erro no portfólio {centrality} (k={k}, ano={year}): {e}")
            all_betas[year][k][centrality] = np.nan

# --- OPCIONAL: Converter resultados para um DataFrame longo (melhor para plotar) ---
rows = []
for year, ks_dict in all_betas.items():
    for k, cents in ks_dict.items():
        for centrality, beta in cents.items():
            rows.append({"Year": year, "k": k, "Centrality": centrality, "Beta": beta})

df_results = pd.DataFrame(rows)
print("\nProcessamento concluído.")
print(df_results.head())

Processando ano: 2015...
Processando ano: 2016...
Processando ano: 2017...
Processando ano: 2018...
Processando ano: 2019...
Processando ano: 2020...
Processando ano: 2021...
Processando ano: 2022...
Processando ano: 2023...
Processando ano: 2024...

Processamento concluído.
   Year   k  Centrality      Beta
0  2015   1     central  1.122735
1  2015   1  peripheral  0.578857
2  2015  10     central  0.980891
3  2015  10  peripheral  0.639678
4  2015  30     central  1.039241


In [2]:
df_results

,Year,k,Centrality,Beta
0,2015,1,central,1.122735
1,2015,1,peripheral,0.578857
2,2015,10,central,0.980891
3,2015,10,peripheral,0.639678
4,2015,30,central,1.039241
5,2015,30,peripheral,0.624696
6,2016,1,central,1.547606
7,2016,1,peripheral,0.465895
8,2016,10,central,1.148203
9,2016,10,peripheral,0.702152


In [4]:
df_results.to_parquet("../../data/07_portfolios_metadata/beta_df.parquet")

In [1]:
import pandas as pd

years = range(2015, 2024)
returns_dict = {}

for year in years:
    for centrality in ["central", "peripheral"]:
        for k in [1, 10, 30]:
            returns_dict[f"{centrality}_{year}_{k}"] = pd.read_csv(f"../../data/06_portfolios/{centrality}_{year}_{k}.csv", index_col="Date")

In [2]:
import yfinance as yf

momentum_dict = {}

for portfolio_name, df_returns in returns_dict.items():
    first_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year-1}-12-29",
        end=f"{year}-01-01"
    )["Close"]

    last_price = yf.download(
        tickers=list(returns_dict[portfolio_name].columns),
        start=f"{year}-12-29",
        end=f"{year+1}-01-01"
    )["Close"]

    momentum_dict[portfolio_name] = last_price.iloc[0] / first_price.iloc[0] - 1

[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed
[*****

KeyboardInterrupt: 

In [ ]:
momentum_df = (
    pd.concat(momentum_dict, names=["portfolio", "Ticker"])
    .reset_index(name=f"momentum12mo_{year}")
)
momentum_df.to_parquet(f"../../data/07_portfolios_metadata/momentum_df_{year}.parquet", index=False)

In [30]:
momentum_df

,portfolio,Ticker,momentum12mo_2024
0,central,AAPL,0.316343
1,central,ACWI,0.177692
2,central,ADI,0.088667
3,central,AMAT,0.017571
4,central,AMKR,-0.204909
...,...,...,...
138,peripheral,VIRC,-0.144975
139,peripheral,WFCF,-0.087085
140,peripheral,WKSP,-0.328859
141,peripheral,XOMA,0.410270


## Completo beta e momentum

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
from tqdm import tqdm

df_metrics = pd.read_parquet("../../data/02_clean/df_metrics.parquet")
tickers = df_metrics['node'].unique().tolist()
years = df_metrics['year'].unique().tolist()

def beta_ols(rp, rm):
    rm = rm.squeeze().rename("rm")
    joined = rp.to_frame("rp").join(rm, how="inner").dropna()
    if len(joined) < 20:  # not enough observations
        return np.nan
    y = joined["rp"]
    X = sm.add_constant(joined["rm"])
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return model.params["rm"]

def momentum_12m(prices):
    """Total return over the year."""
    prices = prices.dropna()
    if len(prices) < 2:
        return np.nan
    return (prices.iloc[-1] / prices.iloc[0]) - 1

records = []

for year in sorted(years):
    print(f"\n=== {year} ===")
    start, end = f"{year}-01-01", f"{year}-12-31"

    # Market returns
    sp500_prices = yf.download("^GSPC", start=start, end=end, progress=False)["Close"]
    sp500_dr = sp500_prices.pct_change().dropna()
    sp500_dr.name = "rm"

    # Tickers to process this year
    tickers_year = df_metrics[df_metrics['year'] == year]['node'].tolist()
    
    # Download all tickers at once
    raw = yf.download(tickers_year, start=start, end=end, progress=False)["Close"]
    if isinstance(raw, pd.Series):
        raw = raw.to_frame(name=tickers_year[0])

    for ticker in tqdm(tickers_year, desc=f"Computing {year}"):
        if ticker not in raw.columns:
            records.append({"node": ticker, "year": year, "beta": np.nan, "momentum": np.nan})
            continue
            
        prices = raw[ticker].dropna()
        returns = prices.pct_change().dropna()

        b = beta_ols(returns, sp500_dr)
        m = momentum_12m(prices)

        records.append({
            "node": ticker,
            "year": year,
            "beta": b,
            "momentum": m
        })

beta_momentum_df = pd.DataFrame(records)
print(beta_momentum_df.describe())


In [ ]:
# Join with df_metrics
df_metrics = df_metrics.merge(beta_momentum_df, on=["node", "year"], how="left")

# Save updated metrics
df_metrics.to_parquet("../../data/02_clean/df_metrics.parquet", index=False)
df_metrics.head()


In [15]:
metadata = pd.read_csv("../../data/02_clean/METADADOS_ATUALIZADO - Sheet1.csv")
metadata["mcap_2024"]

0      3.754818e+12
1      7.254202e+11
2      1.468970e+15
3      6.937964e+11
4      6.758384e+11
           ...     
394    2.054726e+09
395    5.491550e+09
396    5.117357e+10
397    1.722747e+11
398    1.385571e+11
Name: mcap_2024, Length: 399, dtype: float64